In [1]:
# 1. Standard Imports
from scipy.stats import chi2_contingency
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
from dbconfig import engine
"""%load_ext sql
%sql engine
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False"""
print("Environment Ready.")

Environment Ready.


In [11]:
%%sql
select count(*)
from matches
where toss_winner = winner

union all

select count(*)
from matches
where toss_winner != winner

union all

select count(*)
from matches;

,count
0,554
1,536
2,1095


In [7]:
%%sql
select toss_decision, count(*)
from matches
where toss_winner = winner
group by toss_decision;

,toss_decision,count
0,bat,177
1,field,377


In [10]:
%%sql
select
    toss_decision,
    count(*) as total_matches
from matches
group by toss_decision;

,toss_decision,total_matches
0,bat,391
1,field,704


In [12]:
from scipy.stats import chi2_contingency
import numpy as np

contingency = np.array([
    [177, 214],  # Bat: won, lost
    [377, 327]   # Field: won, lost
])

chi2, p, dof, expected = chi2_contingency(contingency)

print(f"Chi-Squared: {chi2}")
print(f"p-value: {p}")
print(f"Degrees of Freedom: {dof}")

Chi-Squared: 6.571677733078204
p-value: 0.01036142309365386
Degrees of Freedom: 1


In [5]:
chi2 = 1.7668181959226748
n = 158  # 164 + 74
k = 2

cramers_v = np.sqrt(chi2 / (n * (k - 1)))
print(f"Cramér's V: {cramers_v}")

Cramér's V: 0.10574683751810368


In [29]:
%%sql
select venue,
       count(distinct id) as match_count,
       sum(case when team1 = winner then 1 else 0 end) as team1_wins,
       round(100.0 * sum(case when team1 = winner then 1 else 0 end) / count(distinct id), 2) as team1_win_pct
from matches
group by venue
order by match_count desc
limit 5;

,venue,match_count,team1_wins,team1_win_pct
0,Wankhede Stadium,118,59,50.00
1,M Chinnaswamy Stadium,94,46,48.94
2,Eden Gardens,93,53,56.99
3,MA Chidambaram Stadium,85,52,61.18
4,"Rajiv Gandhi International Stadium, Uppal",62,31,50.00


In [3]:
%%sql
select venue, count(distinct id) as match_count,
       count(*) filter (where result = 'runs') as batting_result_count,
       count(*) filter (where result = 'wickets') as chasing_result_count,
       round(100.00 * count(*) filter (where result = 'runs') / count(distinct id) ,2) as batting_result_pct,
       round(100.00 * count(*) filter (where result = 'wickets') / count(distinct id) ,2) as chasing_result_pct
from matches
group by venue
order by match_count desc;

,venue,match_count,batting_result_count,chasing_result_count,batting_result_pct,chasing_result_pct
0,Wankhede Stadium,118,53,64,44.92,54.24
1,M Chinnaswamy Stadium,94,41,49,43.62,52.13
2,Eden Gardens,93,40,53,43.01,56.99
3,MA Chidambaram Stadium,85,47,36,55.29,42.35
4,"Rajiv Gandhi International Stadium, Uppal",62,27,34,43.55,54.84
5,"Punjab Cricket Association Stadium, Mohali",61,27,34,44.26,55.74
6,Feroz Shah Kotla,60,28,31,46.67,51.67
7,Sawai Mansingh Stadium,57,20,37,35.09,64.91
8,Dubai International Cricket Stadium,46,21,22,45.65,47.83
9,Dr DY Patil Sports Academy,37,17,20,45.95,54.05


In [8]:

venue_chasing.sort(by = ['match_count','chasing_result_pct'], descending = [True, True]).head(5)

venue,match_count,batting_result_count,chasing_result_count,batting_result_pct,chasing_result_pct
str,i64,i64,i64,f64,f64
"""Wankhede Stadium""",118,53,64,44.92,54.24
"""M Chinnaswamy Stadium""",94,41,49,43.62,52.13
"""Eden Gardens""",93,40,53,43.01,56.99
"""MA Chidambaram Stadium""",85,47,36,55.29,42.35
"""Rajiv Gandhi International Sta…",62,27,34,43.55,54.84


In [9]:
import numpy as np

contingency = np.array([
    [47, 36],  # Bat: won, lost
    [161, 200]   # Field: won, lost
])

chi2, p, dof, expected = chi2_contingency(contingency)

print(f"Chi-Squared: {chi2}")
print(f"p-value: {p}")
print(f"Degrees of Freedom: {dof}")

Chi-Squared: 3.4527850646392624
p-value: 0.06314528493229325
Degrees of Freedom: 1


In [10]:
chi2 = 3.4527850646392624
n = 85  # total Chidambaram matches
k = 2   # categories (runs vs wickets)

cramers_v = (chi2 / (n * (k - 1))) ** 0.5
print(f"Cramér's V: {cramers_v}")

Cramér's V: 0.20154652257099825
